<a href="https://colab.research.google.com/github/GabrielColucciDev/Portifolio/blob/main/Projeto_Analise_Delivery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍔 Projeto: Análise Estratégica de Delivery

---

**Objetivo Principal:**
Realizar uma análise descritiva e exploratória dos dados de pedidos de um delivery. O foco é extrair *insights* acionáveis através do tratamento de dados, engenharia de *features*, análise temporal e cálculo de KPIs estratégicos.

**Tecnologias Utilizadas:**
* **Linguagem:** Python
* **Manipulação e Análise:** Pandas, NumPy
* **Visualização:** Matplotlib, Seaborn
* **Ambiente:** Google Colab

**Contexto do Negócio:**
Atuando como Analista de Dados, o desafio é compreender o comportamento de vendas de um cardápio variado (Salgados, Bebidas, Sobremesas, etc.). Para isso, faremos o cruzamento da base histórica de `pedidos` com as informações de custos e categorias do `cardápio`, visando embasar decisões estratégicas da diretoria.

---

## 1. Importações e Configurações Iniciais

Nesta etapa, preparamos o ecossistema do projeto carregando as ferramentas necessárias. A organização foi dividida em três pilares:

* **Manipulação de Dados:** `pandas` e `numpy` para o processamento tabular e cálculos numéricos avançados.

* **Visualização de Dados:** `matplotlib` e `seaborn` para a futura geração de gráficos estatísticos e painéis visuais.

* **Configuração de Ambiente:** Ajustes globais no Pandas para padronizar a exibição de valores monetários (duas casas decimais), garantir a leitura completa das colunas e a supressão de alertas (`warnings`) para manter o relatório limpo.

In [ ]:
# Bibliotecas para manipulação e análise de dados
import pandas as pd
import numpy as np

# Bibliotecas para visualização de dados (preparando para futuros dashboards)
import matplotlib.pyplot as plt
import seaborn as sns

# Biblioteca para ignorar avisos não críticos (mantém o notebook limpo)
import warnings
warnings.filterwarnings('ignore')

# ---- Configurações Globais do Pandas ----

# 1. Formatar números decimais para 2 casas (ideal para moeda/preços)
pd.set_option('display.float_format', '{:.2f}'.format)

# 2. Garantir que o Pandas mostre todas as colunas, sem truncar com "..."
pd.set_option('display.max_columns', None)

# 3. Estilo padrão para os gráficos do Seaborn (deixa os gráficos mais profissionais)
sns.set_theme(style="whitegrid")

print("Bibliotecas importadas e ambiente configurado com sucesso!")

Bibliotecas importadas e ambiente configurado com sucesso!


---

## 2. Carregamento dos Dados

Nesta etapa, fazemos a importação dos conjuntos de dados (ficheiros CSV) para o nosso ambiente de análise estruturando-os em *DataFrames* do Pandas.

Estamos a utilizar **caminhos relativos** (`dados/nome_do_ficheiro.csv`). Esta é uma boa prática de engenharia de software e análise de dados, pois garante que o projeto seja reprodutível em qualquer máquina que clone o nosso repositório no GitHub, sem a necessidade de reconfigurar caminhos locais absolutos.

* **pedidos:** Registo histórico de todas as transações de vendas.
* **cardapio:** Catálogo de produtos com as respetivas categorias e preços base.

In [ ]:

# Agora o Colab vai encontrar o arquivo neste caminho relativo
pedidos = pd.read_csv('/content/Dados/pedidos.csv')
cardapio = pd.read_csv('/content/Dados/cardapio.csv')

try:
    # Carregamento dos ficheiros CSV para DataFrames
    pedidos = pd.read_csv('/content/Dados/pedidos.csv')
    cardapio = pd.read_csv('/content/Dados/cardapio.csv')

    # Validação visual rápida do carregamento
    print("✅ Dados carregados com sucesso!\n")
    print(f"-> Base de Pedidos: {pedidos.shape[0]} linhas e {pedidos.shape[1]} colunas")
    print(f"-> Base do Cardápio: {cardapio.shape[0]} linhas e {cardapio.shape[1]} colunas")

except FileNotFoundError:
    print("❌ Erro: Os ficheiros não foram encontrados. Certifique-se de que a pasta 'dados' existe no diretório atual com os ficheiros CSV lá dentro.")


✅ Dados carregados com sucesso!

-> Base de Pedidos: 450 linhas e 6 colunas
-> Base do Cardápio: 20 linhas e 3 colunas


---

##  3. Análise Exploratória de Dados (EDA) - Sanity Check

Eu entendo que a **Análise Exploratória de Dados (EDA)** não é apenas uma formalidade matemática, mas sim um diagnóstico clínico de saúde das bases de dados. Então, antes de aplicar modelos de negócios ou agregações financeiras, precisamos entender os limites e falhas estruturais do que foi coletado pelo sistema de analise do Delivery.

###  Cada Função Utilizada:
1. **`head()` e `tail()` (Auditoria Visual de Fronteira):** Validam as extremidades superiores e inferiores das tabelas. Isso nos ajuda a identificar se houve erros de quebra de linha no arquivo CSV ou se as colunas estão desalinhadas.

2. **`info()` (Inspeção de Tipagem e Volumetria):** Mapeia a arquitetura da tabela. Aqui descobrimos se o Pandas interpretou os dados numéricos de forma correta e fazemos o levantamento inicial de **Valores Ausentes (Missing Data)** comparando o número total de linhas com os registros não-nulos de cada coluna.

3. **`describe()` (Estatística Descritiva Descendente):** Expõe métricas de tendência central (média e mediana) e dispersão (desvio padrão, mínimo, máximo). É nossa principal defesa para encontrar *outliers* ou dados impossíveis no negócio (ex: quantidades negativas ou preços zerados).

###  Mapeamento Antecipado da Tabela Dimensão

Nesta célula, estendemos propositalmente a exploração inicial também para a base de `cardapio.csv`. Embora os requisitos formais foquem primariamente na tabela de pedidos, olhar para o cardápio antecipadamente nos permite:
* Identificar visualmente a **chave primária de relacionamento** (`Item`) que unirá as duas tabelas no futuro.
* Compreender a amplitude das categorias de produtos disponíveis para as tomadas de decisões de marketing.

In [ ]:
# Visualização Inicial dos Dados de Pedidos
print("="*50)
print(" PRIMEIRAS LINHAS - BASE DE PEDIDOS")
print("="*50)
display(pedidos.head()) # display() deixei aqui a tabela muito mais bonita no Colab

# Informações Estruturais (Tipos de dados e contagem de nulos)
print("\n" + "="*50)
print(" ESTRUTURA DOS DADOS (INFO) - BASE DE PEDIDOS")
print("="*50)
# O método info() imprime diretamente na tela, por isso não usei display()
pedidos.info()

# Resumo Estatístico das Variáveis Numéricas
print("\n" + "="*50)
print(" RESUMO ESTATÍSTICO (DESCRIBE) - BASE DE PEDIDOS")
print("="*50)
display(pedidos.describe())

# Dando uma olhada rápida também no Cardápio
print("\n" + "="*50)
print(" VISÃO GERAL - BASE DO CARDÁPIO")
print("="*50)
display(cardapio.head())

 PRIMEIRAS LINHAS - BASE DE PEDIDOS


,ID_Pedido,Data,Cliente_ID,Item,Quantidade,Preco_Unitario
0,1,2023-11-16,222,Torta de Limao,23.00,12.93
1,2,2024-05-25,232,Esfiha,NaN,5.87
2,3,2024-12-20,276,Wrap de Frango,7.00,25.98
3,4,2024-10-17,198,Batata Frita,NaN,11.62
4,5,2023-07-06,385,Esfiha,17.00,6.94



 ESTRUTURA DOS DADOS (INFO) - BASE DE PEDIDOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Pedido       450 non-null    int64  
 1   Data            450 non-null    object 
 2   Cliente_ID      450 non-null    int64  
 3   Item            450 non-null    object 
 4   Quantidade      431 non-null    float64
 5   Preco_Unitario  430 non-null    float64
dtypes: float64(2), int64(2), object(2)
memory usage: 21.2+ KB

 RESUMO ESTATÍSTICO (DESCRIBE) - BASE DE PEDIDOS


,ID_Pedido,Cliente_ID,Quantidade,Preco_Unitario
count,450.00,450.00,431.00,430.00
mean,225.50,245.86,15.90,17.86
std,130.05,88.36,9.09,11.60
min,1.00,100.00,1.00,4.09
25%,113.25,169.00,8.00,8.92
50%,225.50,241.50,16.00,13.10
75%,337.75,325.75,24.00,27.40
max,450.00,399.00,30.00,46.07



 VISÃO GERAL - BASE DO CARDÁPIO


,Item,Categoria,Preco_Base
0,Hamburguer,Salgados,22.90
1,Batata Frita,Salgados,12.50
2,Pizza Mussarela,Salgados,39.90
3,Pizza Calabresa,Salgados,42.90
4,Salada Caesar,Saudaveis,28.00


Conclusões do Diagnostico (EDA)



Problema de Qualidade de Dados (Valores Ausentes) - Ao Executar o **info()** e observando as primeiras linhas, chegamos a conclusão mais importante desta etapa: **Nossa base de dados esta "suja"**

- Existem pedidos onde a Quantidade não foi preenchida (em branco/nulo). Um exemplo mais visivel é o "Pedido 2 de Esfiha e o pedido 4 de Batata frita.

- Existem pedidos onde o **Preço_Unitario** esta faltando

**Qual as ações exigidas:** Não podemos calcular o faturamento real do restaurante se faltam quantidades e preços. Isso nos obriga a corrigir essas falhas antes de fazer qualquer matemática.





In [ ]:
#Checagem de Tipos de Memoria
print("--- Tipos de Dados ---")
display(pedidos.dtypes)

--- Tipos de Dados ---


,0
ID_Pedido,int64
Data,object
Cliente_ID,int64
Item,object
Quantidade,float64
Preco_Unitario,float64


In [ ]:
print("\n--- Cardinalidade (Valores Únicos) ---")
print(f"Itens únicos no cardápio: {pedidos['Item'].nunique()}")
print(f"Categorias únicas: {cardapio['Categoria'].unique()}")


--- Cardinalidade (Valores Únicos) ---
Itens únicos no cardápio: 20
Categorias únicas: ['Salgados' 'Saudaveis' 'Bebidas' 'Doces' 'Oriental']


## 4. Tratamento de Valores Ausentes
Antes de prosseguir com os cálculos, precisamos garantir a integridade dos dados.
1. Preencher a `Quantidade` ausente com a média.
2. Remover registros onde o `Preco_Unitario` é nulo.

In [ ]:
# 1. Identificando valores nulos
print("Valores nulos antes do tratamento:")
print(pedidos.isnull().sum())

# 2. Preenchendo a coluna 'Quantidade' com a média
media_quantidade = pedidos['Quantidade'].mean()
pedidos['Quantidade'].fillna(media_quantidade, inplace=True)

# 3. Removendo linhas com valores nulos na coluna 'Preco_Unitario'
pedidos.dropna(subset=['Preco_Unitario'], inplace=True)

print("\nValores nulos após o tratamento:")
print(pedidos.isnull().sum())

Valores nulos antes do tratamento:
ID_Pedido          0
Data               0
Cliente_ID         0
Item               0
Quantidade        19
Preco_Unitario    20
dtype: int64

Valores nulos após o tratamento:
ID_Pedido         0
Data              0
Cliente_ID        0
Item              0
Quantidade        0
Preco_Unitario    0
dtype: int64


/tmp/ipykernel_2047/45426721.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  pedidos['Quantidade'].fillna(media_quantidade, inplace=True)


5. Criação de Novas Colunas (Feature Engineering)
Vamos calcular a receita individual de cada pedido (Quantidade × Preço Unitário).

## 5. Criação de Novas Colunas (Feature Engineering)
Vamos calcular a receita individual de cada pedido (Quantidade × Preço Unitário).# Nova seção

In [ ]:
# Criando a nova coluna de Receita por Item
pedidos['Receita_Item'] = pedidos['Quantidade'] * pedidos['Preco_Unitario']

# Verificando a nova coluna
display(pedidos[['Item', 'Quantidade', 'Preco_Unitario', 'Receita_Item']].head())

,Item,Quantidade,Preco_Unitario,Receita_Item
0,Torta de Limao,23.00,12.93,297.39
1,Esfiha,15.90,5.87,93.35
2,Wrap de Frango,7.00,25.98,181.86
3,Batata Frita,15.90,11.62,184.79
4,Esfiha,17.00,6.94,117.98


## 6. Agregações por Item
Para entender o desempenho do cardápio, vamos agrupar os dados e extrair os produtos campeões de vendas.

In [ ]:
# Agrupando os dados por Item
agrupamento_itens = pedidos.groupby('Item').agg(
    Quantidade_Total=('Quantidade', 'sum'),
    Receita_Total=('Receita_Item', 'sum')
).reset_index()

# Top 5 itens mais vendidos (por quantidade)
top5_quantidade = agrupamento_itens.sort_values(by='Quantidade_Total', ascending=False).head(5)
print("--- Top 5 Itens por Quantidade Vendida ---")
display(top5_quantidade)

# Top 5 itens que geraram mais receita
top5_receita = agrupamento_itens.sort_values(by='Receita_Total', ascending=False).head(5)
print("\n--- Top 5 Itens por Receita Gerada ---")
display(top5_receita)

--- Top 5 Itens por Quantidade Vendida ---


,Item,Quantidade_Total,Receita_Total
7,Hamburguer,510.81,"11,562.23"
3,Brownie,465.90,"4,603.50"
0,Acai 300ml,402.90,"6,502.13"
4,Cafe Expresso,392.00,"2,657.71"
16,Sushi 8 pecas,378.90,"12,085.16"



--- Top 5 Itens por Receita Gerada ---


,Item,Quantidade_Total,Receita_Total
9,Pizza Calabresa,372.81,"15,903.65"
10,Pizza Mussarela,349.00,"14,148.69"
16,Sushi 8 pecas,378.90,"12,085.16"
7,Hamburguer,510.81,"11,562.23"
12,Salada Caesar,352.00,"9,927.96"


## 7. Análise Temporal
Analisando a evolução das vendas mês a mês para encontrar padrões de sazonalidade ou crescimento.

In [ ]:
# Convertendo a coluna 'Data' para o tipo datetime
pedidos['Data'] = pd.to_datetime(pedidos['Data'])

# Extraindo o mês (e ano, para caso os dados passem de 1 ano) para uma nova coluna
pedidos['Mes_Ano'] = pedidos['Data'].dt.to_period('M')

# Calculando a receita total por mês
receita_mensal = pedidos.groupby('Mes_Ano')['Receita_Item'].sum().reset_index()

print("--- Evolução da Receita por Mês ---")
display(receita_mensal)

--- Evolução da Receita por Mês ---


,Mes_Ano,Receita_Item
0,2023-01,"1,988.58"
1,2023-02,"6,646.03"
2,2023-03,"3,751.15"
3,2023-04,"1,395.98"
4,2023-05,"4,297.77"
5,2023-06,"5,273.88"
6,2023-07,"4,692.73"
7,2023-08,"4,256.75"
8,2023-09,"5,872.16"
9,2023-10,"2,999.95"


## 8. Integração de Dados (Merge)
Enriquecendo nossos dados de pedidos com as informações de Categoria contidas no arquivo `cardapio.csv`.

In [ ]:
# Realizando o left merge usando a coluna 'Item'
df_completo = pd.merge(pedidos, cardapio, on='Item', how='left')

# Calculando a receita total por categoria
receita_categoria = df_completo.groupby('Categoria')['Receita_Item'].sum().reset_index()
receita_categoria = receita_categoria.sort_values(by='Receita_Item', ascending=False)

print("--- Receita Total por Categoria ---")
display(receita_categoria)

--- Receita Total por Categoria ---


,Categoria,Receita_Item
3,Salgados,"53,073.30"
1,Doces,"22,293.94"
2,Oriental,"21,599.80"
4,Saudaveis,"16,539.04"
0,Bebidas,"9,146.51"


## 9. Filtros e Consultas Específicas
Consultando uma regra de negócio específica: Pedidos da categoria 'Salgados' em grandes volumes (quantidade > 10).

In [ ]:
# Aplicando múltiplos filtros
filtro_salgados_volume = df_completo[(df_completo['Categoria'] == 'Salgados') & (df_completo['Quantidade'] > 10)]

print("--- Pedidos de Salgados com mais de 10 unidades ---")
display(filtro_salgados_volume.head(10)) # Mostrando apenas os primeiros 10 para visualização limpa

--- Pedidos de Salgados com mais de 10 unidades ---


,ID_Pedido,Data,Cliente_ID,Item,Quantidade,Preco_Unitario,Receita_Item,Mes_Ano,Categoria,Preco_Base
1,2,2024-05-25,232,Esfiha,15.90,5.87,93.35,2024-05,Salgados,6.50
3,4,2024-10-17,198,Batata Frita,15.90,11.62,184.79,2024-10,Salgados,12.50
4,5,2023-07-06,385,Esfiha,17.00,6.94,117.98,2023-07,Salgados,6.50
5,6,2023-02-04,260,Empanada,16.00,9.48,151.68,2023-02,Salgados,8.90
14,15,2023-02-03,103,Batata Frita,14.00,13.13,183.82,2023-02,Salgados,12.50
15,16,2025-05-28,213,Batata Frita,11.00,13.18,144.98,2025-05,Salgados,12.50
23,25,2023-04-20,342,Esfiha,20.00,6.84,136.80,2023-04,Salgados,6.50
32,34,2025-09-12,182,Pizza Calabresa,28.00,39.83,"1,115.24",2025-09,Salgados,42.90
33,35,2023-10-06,354,Hamburguer,24.00,20.78,498.72,2023-10,Salgados,22.90
36,38,2023-09-12,343,Hamburguer,18.00,24.96,449.28,2023-09,Salgados,22.90


## 10. KPIs e Análise Estatística com NumPy
Cálculo dos indicadores chaves de performance do delivery e estatística descritiva de distribuição com NumPy.

In [ ]:
# 1. Receita Total
receita_total_negocio = df_completo['Receita_Item'].sum()

# 2. Total de Itens Vendidos
total_itens_vendidos = df_completo['Quantidade'].sum()

# 3. Ticket Médio (Receita Total / Número de Pedidos Únicos)
numero_pedidos = df_completo['ID_Pedido'].nunique()
ticket_medio = receita_total_negocio / numero_pedidos

print("========== KPIs DO NEGÓCIO ==========")
print(f"Receita Total: R$ {receita_total_negocio:,.2f}")
print(f"Total de Itens Vendidos: {total_itens_vendidos:,.0f} unidades")
print(f"Ticket Médio: R$ {ticket_medio:,.2f} por pedido")
print("=====================================\n")

# 4. Cálculo de Percentis usando NumPy
percentis = [25, 50, 75]

# Percentis para Preço Unitário
percentis_preco = np.percentile(df_completo['Preco_Unitario'], percentis)
# Percentis para Quantidade
percentis_qtd = np.percentile(df_completo['Quantidade'], percentis)

print("--- Percentis (25%, 50%, 75%) ---")
print(f"Preço Unitário: 25% = R${percentis_preco[0]:.2f} | 50% (Mediana) = R${percentis_preco[1]:.2f} | 75% = R${percentis_preco[2]:.2f}")
print(f"Quantidade:     25% = {percentis_qtd[0]:.0f} un | 50% (Mediana) = {percentis_qtd[1]:.0f} un | 75% = {percentis_qtd[2]:.0f} un")

========== KPIs DO NEGÓCIO ==========
Receita Total: R$ 122,652.59
Total de Itens Vendidos: 6,833 unidades
Ticket Médio: R$ 285.24 por pedido

--- Percentis (25%, 50%, 75%) ---
Preço Unitário: 25% = R$8.92 | 50% (Mediana) = R$13.10 | 75% = R$27.40
Quantidade:     25% = 8 un | 50% (Mediana) = 16 un | 75% = 24 un
